### N-gram + Aspect(속성) 분류 분석 (국내 KR / 글로벌 GB)

## N-gram + Aspect 분석 개요

TF-IDF 분석에서 한 단계 더 나아가, **N-gram(1~2/1~3 단어 조합)** 으로 문맥을 살리고 각 키워드를 **속성(Aspect) 카테고리**로 분류해 "어떤 속성이 긍정/부정에서 부각되는지"를 파악하는 분석입니다.

### 처리 순서
1. 확신도 0.8 이상 리뷰 필터링
2. **속성사전(`ASPECT_RULES`)** 정의 — 발림성, 끈적임, 백탁, 보습력 등 속성별로 관련 표현을 묶음
3. N-gram TF-IDF 계산 (KR: 1~2gram / GB: 1~3gram)
4. 추출된 키워드를 속성사전 기준으로 분류하여 "속성별 상위 키워드"로 정리
5. 제품별 · 전체 리뷰 기준으로 반복 실행 후 결과 저장

### ⚠️ 참고 및 실행 전 준비사항
- 속성사전(`ASPECT_RULES`, `en_norm_groups`)은 리뷰 원문을 반복 탐색하며 만든 결과물입니다. 이 사전을 만드는 과정(특정 단어·리뷰 검색, 빈도 확인)은 본 노트북에서는 생략하고, **최종적으로 사용한 사전과 실제 분석 로직만 정리**했습니다.
- 불용어 사전 파일(`stopwords.txt`, `stopwords_gb.txt`)이 별도로 필요합니다.
- **본 노트북 실행 결과로 생성되는 CSV(`kr_ngram_tfidf.csv`, `gb_ngram_tfidf.csv`)는 리포지토리에 포함하지 않습니다.** 분석 코드와 방법론 참고용으로만 공개하며, 실제 산출 데이터는 별도로 관리합니다.
- 한국어 형태소 분석(KoNLPy Okt)은 Java(JDK) 설치가 필요하고, 영어 처리(NLTK)는 최초 1회 리소스 다운로드가 필요합니다.


### 전체 리뷰 통합(제품 구분 없음) 분석 관련 참고
제품 구분 없이 전체 리뷰를 긍정/부정으로 나눠 2-gram TF-IDF를 계산합니다 (`min_df=3`, 중복 표현·공통 키워드는 자동 정리).
다만 발표자료의 최종 Top 5는 여기서 한 번 더 수동으로 검토하여, 같은 단어가 반복되지 않도록 선별한 결과이며 이 단계는 코드로 자동화되어 있지 않습니다.

## 1. 국내(KR) 리뷰 N-gram + Aspect 분석

In [ ]:
#step 1. 모듈
from konlpy.tag import Okt
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

okt = Okt()

#step 2. 파일 불러오기 (경로는 실행 환경에 맞게 수정)
kr = pd.read_csv('kr_sentiment.csv')

#step 3. 확신도 0.8 기준으로 필터링
kr_filtered = kr[kr['sentiment_score'] >= 0.8].copy()

# step 4. 상품 코드 → 상품명 매핑 (예시. 실제 프로젝트에서는 10개 상품의 goods_no를 매핑하여 사용)
product_map = {
    'A000000000001': '1_product_a',
    'A000000000002': '2_product_b',
    # ... 나머지 상품 매핑 생략
}
kr_filtered['product_name'] = kr_filtered['goods_no'].map(product_map)


In [ ]:
# 속성사전(Aspect Rules) 정의 — 반복 탐색 끝에 확정된 최종 버전
# 20개 속성(발림성, 끈적임, 백탁, 보습력 등)별로 관련 표현을 묶어, TF-IDF로 뽑힌 키워드를
# 어떤 속성에 대한 언급인지 분류하는 기준으로 사용
ASPECT_RULES = {
    '발림성' : {'발라', '발랏', '발랏는', '발랏는데', '발랏다', '발랏을때', '발랏을땬', '발랴', '발리',
                '발리나', '발린', '발림', '발링', '뻑벅한', '뻑뻑', '뻑하거', '밀리', '밀어내기'},
    '눈시림' : {'시려웠', '시렵거', '시렵던', '시리', '시림'},
    '밀착력' : {'밀착', '밀착렫', '밀철'},
    '끈적임' : {'끈', '끈끈', '끈끈함', '끈덕', '끈덕끈', '끈쩍', '끈쩍끈쩍', '끈쩍임', '적임'},
    '화끈거림' : {'화끈', '화끈거렸', '후끈'},
    '백탁'   : {'탁', '백탁현상', '백탁'},
    '가성비' : {'가성', '저렴'},
    '무기자차' : {'무기'},
    '유기자차' : {'유기'},
    '유지력' : {'유지', '유지됨', '지속'},
    '흡수력' : {'흡수'},
    '트러블' : {'뽀로지', '뾰루지', '뽀루지'},
    '피부타입' : {'지성만', '지성은', '지성이면', '지성인', '부지', '건성', '복합'},
    '쿨링감' : {'쿨롱', '쿨링', '쿨링금'},
    '보습력' : {'촉촉', '수분', '보습'}
}

# ASPECT_RULES → lookup용 딕셔너리로 변환
norm_dict = {}
for target, sources in ASPECT_RULES.items():
    for source in sources:
        norm_dict[source] = target

# 불용어 사전 (별도 파일 필요 - 경로는 실행 환경에 맞게 수정)
with open('stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())
stopwords = {w for w in stopwords if w.strip()}


### N-gram 분석

In [ ]:
#N-gram 토크나이저 (명사+형용사 추출, 정규화, 불용어 제거)
def ngram_tokenizer(text):
    morphs = okt.pos(str(text), stem=True, norm=True)
    tokens = []
    for word, pos in morphs:
        if pos in ['Noun', 'Adjective']:
            word = norm_dict.get(word, word)
            if word not in stopwords:
                tokens.append(word)
    return tokens

#Aspect 분류 함수 — 키워드가 어떤 속성에 해당하는지 판별
def classify_aspect(term):
    hits = []
    for aspect, keywords in ASPECT_RULES.items():
        if any(kw in term for kw in keywords):
            hits.append(aspect)
    return hits if hits else ['기타']

#제품별 N-gram TF-IDF + Aspect 분석 함수
def analyze_ngram_aspect(product_name, top_n=30):
    print(f"\n{'-'*30}")
    print(f'제품 : {product_name}')
    print(f"\n{'-'*30}")

    df = kr_filtered[kr_filtered['product_name'] == product_name]
    pos_docs = df[df['sentiment_label'] == '긍정']['review_clean'].tolist()
    neg_docs = df[df['sentiment_label'] == '부정']['review_clean'].tolist()

    results = {}

    for sentiment, docs in [('긍정', pos_docs), ('부정', neg_docs)]:
        if not docs:
            continue

        # TF-IDF (1~2gram)
        vectorizer = TfidfVectorizer(
            tokenizer=ngram_tokenizer,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            max_df=0.9,
            min_df=2
        )
        tfidf_matrix = vectorizer.fit_transform(docs)
        feature_names = vectorizer.get_feature_names_out()
        avg_scores = np.mean(tfidf_matrix.toarray(), axis=0)
        df_tfidf = pd.Series(avg_scores, index=feature_names)\
                     .sort_values(ascending=False)\
                     .head(top_n)

        # Aspect 분류
        aspect_result = {}
        for term, score in df_tfidf.items():
            aspects = classify_aspect(term)
            for aspect in aspects:
                if aspect not in aspect_result:
                    aspect_result[aspect] = []
                aspect_result[aspect].append((term, round(score, 4)))
        results[sentiment] = aspect_result

        # 출력
        print(f'\n --- {sentiment} 리뷰 속성별 키워드 ---')
        for aspect, terms in sorted(aspect_result.items()):
            if aspect == '기타':
                continue
            terms_str = ','.join([f'{t}({s})' for t, s in terms[:3]])
            print(f' [{aspect}] {terms_str}')

    return results


In [ ]:
# 실행 (예시 3개 상품)
target_products = ['1_product_a', '2_product_b']  # 나머지 상품 생략
all_ngram_results = {}

for product in target_products:
    result = analyze_ngram_aspect(product, top_n=30)
    all_ngram_results[product] = result


In [ ]:
# 전체 10개 제품 N-gram 분석 + 결과 저장
target_products = ['1_product_a', '2_product_b']  # 나머지 상품 생략

all_results = []

for product in target_products:
    for sentiment in ['긍정', '부정']:
        df_p = kr_filtered[kr_filtered['product_name'] == product]
        docs = df_p[df_p['sentiment_label'] == sentiment]['review_clean'].tolist()

        if len(docs) < 5:  # 리뷰 너무 적으면 스킵
            print(f'⚠️ {product} {sentiment} 리뷰 부족 ({len(docs)}건) → 스킵')
            continue

        vectorizer = TfidfVectorizer(
            tokenizer=ngram_tokenizer,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 2),
            max_df=0.9,
            min_df=2,
        )
        tfidf_matrix = vectorizer.fit_transform(docs)
        feature_names = vectorizer.get_feature_names_out()
        avg_scores = np.mean(tfidf_matrix.toarray(), axis=0)

        df_result = pd.Series(avg_scores, index=feature_names)\
                      .sort_values(ascending=False)\
                      .head(30)\
                      .reset_index()
        df_result.columns = ['term', 'score']
        df_result['product_name'] = product
        df_result['sentiment'] = sentiment
        df_result['rank'] = range(1, len(df_result) + 1)

        all_results.append(df_result)
        print(f'✅ {product} {sentiment} 완료 ({len(docs)}건)')

# 전체 결과 합치기
df_ngram_all = pd.concat(all_results, ignore_index=True)
df_ngram_all = df_ngram_all[['product_name', 'sentiment', 'rank', 'term', 'score']]

# 저장 (결과 파일은 리포지토리에는 포함하지 않음 - 로컬 분석용)
df_ngram_all.to_csv('kr_ngram_tfidf.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ 저장 완료: kr_ngram_tfidf.csv ({len(df_ngram_all)}행)')

# 확인
print('\n=== 샘플 출력 ===')
print(df_ngram_all.head(20).to_string(index=False))


### 전체 리뷰 통합(제품 구분 없음) 2-gram 분석 — KR

In [ ]:
# ── 아래 두 함수는 KR/GB 전체 통합 분석에서 공통으로 사용 ──

def dedup_mirror(series, top_k=30):
    """
    자기반복(예: '끈적임 끈적임')과 순서만 다른 중복 표현(예: '발림성 촉촉하다' / '촉촉하다 발림성')을
    값이 높은 쪽 하나로 통합
    """
    seen = {}
    for term, score in series.items():
        parts = term.split()
        if len(parts) == 2 and parts[0] == parts[1]:
            continue
        key = tuple(sorted(parts))
        if key not in seen or seen[key][1] < score:
            seen[key] = (term, score)
    ranked = sorted(seen.values(), key=lambda x: -x[1])[:top_k]
    return {tuple(sorted(t.split())): (t, s) for t, s in ranked}

def resolve_common_and_pick(pos_dict, neg_dict, top_n=10):
    """
    긍정/부정에 공통으로 등장하는 표현은 값이 더 높은 쪽에만 귀속
    """
    common_keys = set(pos_dict) & set(neg_dict)
    pos_final = {k: v for k, v in pos_dict.items() if k not in common_keys or pos_dict[k][1] >= neg_dict[k][1]}
    neg_final = {k: v for k, v in neg_dict.items() if k not in common_keys or neg_dict[k][1] >= pos_dict[k][1]}
    pos_sorted = sorted(pos_final.values(), key=lambda x: -x[1])[:top_n]
    neg_sorted = sorted(neg_final.values(), key=lambda x: -x[1])[:top_n]
    return pos_sorted, neg_sorted


In [ ]:
# 전체 리뷰(제품 구분 없음)를 긍정/부정으로 나눠 2-gram TF-IDF 계산
def get_bigram_ranking(docs, min_df=3, pool=30):
    vectorizer = TfidfVectorizer(
        tokenizer=ngram_tokenizer,
        token_pattern=None,
        lowercase=False,
        ngram_range=(2, 2),   # 2-gram만
        max_df=0.9,
        min_df=min_df
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    avg_scores = np.mean(tfidf_matrix.toarray(), axis=0)
    series = pd.Series(avg_scores, index=feature_names).sort_values(ascending=False).head(pool)
    return dedup_mirror(series, top_k=pool)

pos_docs_all = kr_filtered[kr_filtered['sentiment_label'] == '긍정']['review_clean'].tolist()
neg_docs_all = kr_filtered[kr_filtered['sentiment_label'] == '부정']['review_clean'].tolist()

pos_ranked = get_bigram_ranking(pos_docs_all, min_df=3, pool=30)
neg_ranked = get_bigram_ranking(neg_docs_all, min_df=3, pool=30)

pos_final, neg_final = resolve_common_and_pick(pos_ranked, neg_ranked, top_n=10)

print('[전체 국내 리뷰 — 긍정 2-gram Top 10 (자동 랭킹, 최종 Top 5는 수동 검토 필요)]')
for term, score in pos_final:
    print(f'  {term}: {score:.4f}')

print('\n[전체 국내 리뷰 — 부정 2-gram Top 10 (자동 랭킹, 최종 Top 5는 수동 검토 필요)]')
for term, score in neg_final:
    print(f'  {term}: {score:.4f}')


## 2. 글로벌(GB) 리뷰 N-gram + Aspect 분석

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

lemmatizer = WordNetLemmatizer()

# 파일 불러오기 (경로는 실행 환경에 맞게 수정)
gb = pd.read_csv('gb_sentiment.csv')
gb = gb[gb['review_clean'].notna()]
gb = gb[gb['review_clean'] != ''].reset_index(drop=True)

# 확신도 0.8 이상 필터링
gb_filtered = gb[gb['sentiment_score'] >= 0.8].copy()
print(f'전처리 후: {len(gb_filtered):,}건')
print(gb_filtered['sentiment_label'].value_counts())

# 상품 코드 → 상품명 매핑 (예시. 실제 프로젝트에서는 10개 상품의 prdtNo를 매핑하여 사용)
product_map_gb = {
    'GA000000001': '1_product_a',
    'GA000000002': '2_product_b',
    # ... 나머지 상품 매핑 생략
}
gb_filtered['product_name'] = gb_filtered['goods_no'].map(product_map_gb)


In [ ]:
# 불용어 사전 (별도 파일 필요 - 경로는 실행 환경에 맞게 수정)
with open('stopwords_gb.txt', 'r', encoding='utf-8') as f:
    base_stopwords = set(f.read().splitlines())
    base_stopwords = {w.strip() for w in base_stopwords if w.strip()}

print(f'base 불용어: {len(base_stopwords)}개')

# 영어 정규화 사전 (반복 탐색 끝에 확정된 최종 버전)
en_norm_groups = {
    'white_cast'   : {'white', 'cast', 'whitecast', 'casting', 'overcast'},
    'lightweight'  : {'light', 'lightweighted', 'litghweight', 'weightless'},
    'sticky'       : {'stickiness', 'tacky', 'stick', 'sticker', 'stickey', 'stickness'},
    'greasy'       : {'greasiness', 'grease'},
    'moisturizing' : {'moisturizes', 'moisturizer', 'moisturising', 'moisturiser', 'moisture', 'moist'},
    'hydrating'    : {'hydration', 'hydrated'},
    'scent'        : {'smell', 'fragrance'},
    'acne'         : {'breakout', 'pimple', 'blemish', 'prone'},
    'irritation'   : {'irritate', 'irritated', 'irritant', 'irritates', 'irritating'},
    'packaging'    : {'package', 'pack'},
    'smooth'       : {'smoothly', 'spread', 'blend'},
    'oily'         : {'oil', 'oiler', 'oiliness', 'oily'},
    'transparency' : {'transparent'},
    'brightening'  : {'bright', 'brightens', 'brighter', 'brightness'},
    'breakout'     : {'break', 'breakouts', 'outbreak'},
    'absorption'   : {'absorb', 'absorbance', 'absorbed', 'absorbency', 'absorbent',
                       'absorber', 'absorbes', 'absorbing', 'absorbs'}
}

en_norm_dict = {}
for target, sources in en_norm_groups.items():
    for source in sources:
        en_norm_dict[source] = target


In [ ]:
# N-gram 토크나이저 (영어 - base 불용어만 적용)
def en_ngram_tokenizer(text):
    tokens = word_tokenize(str(text).lower())
    tagged = pos_tag(tokens)
    result = []
    for word, tag in tagged:
        if tag.startswith('NN') or tag.startswith('JJ'):
            if tag.startswith('NN'):
                lemma = lemmatizer.lemmatize(word, pos='n')
            else:
                lemma = lemmatizer.lemmatize(word, pos='a')
            lemma = en_norm_dict.get(lemma, lemma)
            if len(lemma) >= 2 and lemma not in base_stopwords:
                result.append(lemma)
    return result


In [ ]:
# 전체 10개 제품 N-gram 분석 + 저장
target_products_gb = ['1_product_a', '2_product_b']  # 나머지 상품 생략

# 의미 없는 단순 감성 표현(1-gram)은 결과에서 제외하기 위한 참고용 목록
sentiment_1gram = {
    'good', 'bad', 'best', 'great', 'nice', 'perfect',
    'excellent', 'amazing', 'love', 'favourite', 'favorite',
    'happy', 'glad', 'horrible', 'terrible', 'awful'
}

def is_valid_term(term):
    tokens = term.split()
    # 동일 단어가 반복된 중복 2-gram 제거 (예: 'good good')
    if len(tokens) >= 2 and len(set(tokens)) == 1:
        return False
    return True

all_results_gb_ngram = []

for product in target_products_gb:
    for sentiment in ['긍정', '부정']:
        df_p = gb_filtered[gb_filtered['product_name'] == product]
        docs = df_p[df_p['sentiment_label'] == sentiment]['review_clean'].tolist()

        if len(docs) < 5:
            print(f'⚠️ {product} {sentiment} 리뷰 부족 ({len(docs)}건) → 스킵')
            continue

        vectorizer = TfidfVectorizer(
            tokenizer=en_ngram_tokenizer,
            token_pattern=None,
            lowercase=False,
            ngram_range=(1, 3),
            max_df=0.9,
            min_df=1,
        )
        tfidf_matrix = vectorizer.fit_transform(docs)
        feature_names = vectorizer.get_feature_names_out()
        avg_scores = np.mean(tfidf_matrix.toarray(), axis=0)

        df_result = pd.Series(avg_scores, index=feature_names)\
                      .sort_values(ascending=False)\
                      .reset_index()
        df_result.columns = ['term', 'score']
        df_result = df_result[df_result['term'].apply(is_valid_term)]
        df_result = df_result.head(30)
        df_result['product_name'] = product
        df_result['sentiment'] = sentiment
        df_result['rank'] = range(1, len(df_result) + 1)

        all_results_gb_ngram.append(df_result)
        print(f'✅ {product} {sentiment} 완료 ({len(docs)}건)')

# 결과 합치기 + 저장 (결과 파일은 리포지토리에는 포함하지 않음 - 로컬 분석용)
df_ngram_gb = pd.concat(all_results_gb_ngram, ignore_index=True)
df_ngram_gb = df_ngram_gb[['product_name', 'sentiment', 'rank', 'term', 'score']]

df_ngram_gb.to_csv('gb_ngram_tfidf.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ 저장 완료: gb_ngram_tfidf.csv ({len(df_ngram_gb)}행)')

print('\n=== 샘플 출력 ===')
print(df_ngram_gb.head(20).to_string(index=False))


### 전체 리뷰 통합(제품 구분 없음) 2-gram 분석 — GB

In [ ]:
# 전체 리뷰(제품 구분 없음)를 긍정/부정으로 나눠 2-gram TF-IDF 계산
def get_bigram_ranking_en(docs, min_df=3, pool=30):
    vectorizer = TfidfVectorizer(
        tokenizer=en_ngram_tokenizer,
        token_pattern=None,
        lowercase=False,
        ngram_range=(2, 2),   # 2-gram만
        max_df=0.9,
        min_df=min_df
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    avg_scores = np.mean(tfidf_matrix.toarray(), axis=0)
    series = pd.Series(avg_scores, index=feature_names).sort_values(ascending=False).head(pool)
    return dedup_mirror(series, top_k=pool)

pos_docs_all = gb_filtered[gb_filtered['sentiment_label'] == '긍정']['review_clean'].tolist()
neg_docs_all = gb_filtered[gb_filtered['sentiment_label'] == '부정']['review_clean'].tolist()

pos_ranked = get_bigram_ranking_en(pos_docs_all, min_df=3, pool=30)
neg_ranked = get_bigram_ranking_en(neg_docs_all, min_df=3, pool=30)

pos_final, neg_final = resolve_common_and_pick(pos_ranked, neg_ranked, top_n=10)

print('[전체 글로벌 리뷰 — 긍정 2-gram Top 10 (자동 랭킹, 최종 Top 5는 수동 검토 필요)]')
for term, score in pos_final:
    print(f'  {term}: {score:.4f}')

print('\n[전체 글로벌 리뷰 — 부정 2-gram Top 10 (자동 랭킹, 최종 Top 5는 수동 검토 필요)]')
for term, score in neg_final:
    print(f'  {term}: {score:.4f}')
